In [ ]:
import os
import pandas as pd
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt

In [6]:
DATA_DIR = "CUB_200_2011"
IMAGES_DIR = os.path.join(DATA_DIR, "images")

# Load the metadata files
images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["image_id", "image_path"]
)

labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["image_id", "class_id"]
)

bbox_df = pd.read_csv(
    os.path.join(DATA_DIR, "bounding_boxes.txt"),
    sep=" ",
    names=["image_id", "x", "y", "width", "height"]
)

splt_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["image_id", "is_train"]
)

# Merge all the dataframes into a single frame called 'data'
data = images_df.merge(labels_df, on="image_id").merge(bbox_df, on="image_id").merge(splt_df, on="image_id")

unique_classes = sorted(data["class_id"].unique())
print(f"Number of unique classes: {len(unique_classes)}")

class_mapping = {old: new for new, old in enumerate(unique_classes)}

data["label"] = data["class_id"].map(class_mapping)

data["full_path"] = data["image_path"].apply(
    lambda x: os.path.join(IMAGES_DIR, x)
)

# Split data into training and testing sets
train_df = data[data["is_train"] == 1].reset_index(drop=True)
test_df = data[data["is_train"] == 0].reset_index(drop=True)

train_paths = train_df["full_path"].values
train_labels = train_df["label"].values
train_bboxes = train_df[["x", "y", "width", "height"]].values

test_paths = test_df["full_path"].values
test_labels = test_df["label"].values
test_bboxes = test_df[["x", "y", "width", "height"]].values

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))
print("Number of classes:", data["label"].nunique())

Number of unique classes: 200
Training samples: 5994
Testing samples: 5794
Number of classes: 200


In [ ]:
# This function extracts HOG feature from an image
def extract_hog(image_path):
    # Load image and convert to RGB
    image = Image.open(image_path).convert("RGB")
    
    # Resize to fixed size for consistent feature extraction
    image = resize(np.array(image), (128, 128))
    
    # Convert to grayscale (HOG works on intensity gradients)
    gray = rgb2gray(image)

    # Extract HOG features (edge/shape descriptors)
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )
    return features

# This function builds the dataset (X, y)
def build_dataset(paths, labels):
    X, y = [], []
    for p, l in zip(paths, labels):
        # Extract feature for each image
        X.append(extract_hog(p))
        y.append(l)
    return np.array(X), np.array(y)

# Build training and testing feature sets
X_train, y_train = build_dataset(train_paths, train_labels)
X_test, y_test = build_dataset(test_paths, test_labels)

# Train SVM classifier (linear kernel works well for HOG)
model = SVC(kernel="linear")
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate performance
print("Experiment 1 (Whole Image + HOG + SVM) Accuracy:", accuracy_score(y_test, y_pred))

# You should also calculate the confusion matrix and visualise some correct/incorrect samples to reason the results.



Experiment 1 (Whole Image + HOG + SVM) Accuracy: 0.03296513634794615


In [ ]:
#

Number of correct predictions: 191
Correctly classified: CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_0085_92.jpg (Label: 0)
Correctly classified: CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_0053_796109.jpg (Label: 0)
Correctly classified: CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_0037_796120.jpg (Label: 0)
Correctly classified: CUB_200_2011/images/005.Crested_Auklet/Crested_Auklet_0036_794905.jpg (Label: 4)
Correctly classified: CUB_200_2011/images/006.Least_Auklet/Least_Auklet_0025_795087.jpg (Label: 5)
Number of incorrect predictions: 5603
Incorrectly classified: CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_0046_18.jpg (True Label: 0, Predicted: 137)
Incorrectly classified: CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_0002_55.jpg (True Label: 0, Predicted: 52)
Incorrectly classified: CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_002